# Exercise 5, Snowball Earth and the ice&ndash;albedo feedback

These exercises go with **Lecture 5 &mdash; Snowball Earth**. They reuse its energy
balance model with a temperature-dependent albedo, reproduced in the setup cell, and
explore the hysteresis loop from three directions: CO$_2$, the strength of the
ice&ndash;albedo contrast, and a single seasonal melt.

Fill only the cells marked

```python
# ==== YOUR CODE ====
```


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def calc_alpha(T, alpha0=0.3, alphai=0.5, dT=10.0):
    """Planetary albedo: alpha0 (ice-free) -> alphai (frozen) as T drops through +/- dT."""
    if T < -dT:
        return alphai
    if T >= dT:
        return alpha0
    return alphai + (alpha0 - alphai) * (T + dT) / (2.0 * dT)


class ebm:
    """Zero-order EBM with an ice-albedo feedback (Lecture 5)."""

    def __init__(self, T, S=1368.0, co2=280.0, alphai=0.5, deltat=1.0):
        self.T = float(T)
        self.S = S
        self.co2 = co2
        self.alphai = alphai
        self.deltat = deltat
        self.C = 51.0
        self.a = 5.0
        self.A = 221.2
        self.B = -1.3
        self.co2_pi = 280.0

    def tendency(self):
        alpha = calc_alpha(self.T, alphai=self.alphai)
        ASR = self.S * (1 - alpha) / 4.0
        OTR = self.A - self.B * self.T
        GH = self.a * np.log(self.co2 / self.co2_pi)
        return (ASR - OTR + GH) / self.C

    def run(self, years):
        for _ in range(int(years)):
            self.T = self.T + self.deltat * self.tendency()
        return self.T


def equilibrium_T(T_start, S=1368.0, co2=280.0, alphai=0.5, years=600):
    """Integrate to steady state and return the final temperature [degC]."""
    return ebm(T_start, S=S, co2=co2, alphai=alphai).run(years)


print("warm start ->", round(equilibrium_T(20.0), 2), "degC")
print("cold start ->", round(equilibrium_T(-40.0), 2), "degC   (two stable climates)")


## Exercise 1 &mdash; the bifurcation diagram in CO$_2$

Lecture 5 traced the hysteresis loop by sweeping the solar constant $S$. Here you sweep
**CO$_2$** instead, at fixed $S = 1368$ W m⁻², and find the two thresholds: the CO$_2$
needed to **deglaciate** a Snowball, and the (much lower) CO$_2$ at which an ice-free
planet **re-freezes**.

**Your task.**
1. Sweep CO$_2$ **upward** on a log grid from 280 ppm to $10^6$ ppm, always restarting
   from the *previous* equilibrium temperature (so you stay on the cold branch until it
   disappears).
2. Sweep CO$_2$ **downward** over the same grid, restarting from the previous
   equilibrium (staying on the warm branch).
3. The plot overlays both; read off the two threshold CO$_2$ values.

**Hints.**
* `co2_grid = np.logspace(np.log10(280), np.log10(1e6), 60)`.
* Up-sweep: `T = -45.0` before the loop; inside, `T = equilibrium_T(T, co2=c)` and store.
* Down-sweep: iterate over `co2_grid[::-1]`, starting from `T = 15.0`.
* Both thresholds are where consecutive stored temperatures jump. After putting
  `T_down` back in ascending-CO$_2$ order, the warm-to-cold drop also shows up as a big
  positive `np.diff`, so use `np.argmax(np.diff(...))` for both.


In [ ]:
co2_grid = np.logspace(np.log10(280), np.log10(1e6), 60)

# ==== YOUR CODE ====
T_up = []
T = -45.0
for c in co2_grid:
    # T = equilibrium_T(T, co2=c)
    # T_up.append(T)
    pass
T_up = np.array(T_up)

T_down = []
T = 15.0
for c in co2_grid[::-1]:
    # T = equilibrium_T(T, co2=c)
    # T_down.append(T)
    pass
T_down = np.array(T_down[::-1])            # put back in ascending-CO2 order

co2_deglaciate = None      # <-- co2_grid[np.argmax(np.diff(T_up))]
co2_refreeze   = None      # <-- co2_grid[np.argmax(np.diff(T_down))]
# ===================

print(f"deglaciation threshold : {co2_deglaciate}")
print(f"re-freezing threshold  : {co2_refreeze}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(8, 5))
if len(T_up):
    ax.semilogx(co2_grid, T_up, "o-", color="tab:blue", ms=3, label="CO$_2$ increasing (cold branch)")
if len(T_down):
    ax.semilogx(co2_grid, T_down, "o-", color="tab:red", ms=3, label="CO$_2$ decreasing (warm branch)")
if co2_deglaciate:
    ax.axvline(co2_deglaciate, color="tab:blue", ls="--", lw=1)
if co2_refreeze:
    ax.axvline(co2_refreeze, color="tab:red", ls="--", lw=1)
ax.axvline(280, color="0.5", ls=":", label="pre-industrial")
ax.set_xlabel("atmospheric CO$_2$ [ppm]"); ax.set_ylabel("equilibrium temperature [degC]")
ax.set_title("Hysteresis of the Snowball in CO$_2$")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
fig.tight_layout()


```{admonition} Answers
:class: note
1. What is the deglaciation CO$_2$, in ppm and as a multiple of pre-industrial? Why is
   so much CO$_2$ needed to melt a Snowball (think about what the albedo is doing to the
   absorbed sunlight)?
2. The re-freezing threshold is far *below* the deglaciation one. Sketch the loop and
   mark the range of CO$_2$ over which **two** stable climates exist.
3. Geologists think Snowball Earth ended when volcanic CO$_2$ built up over millions of
   years. Does your threshold support that story?
```


## Exercise 2 &mdash; how the hysteresis width depends on the ice&ndash;albedo contrast

The whole loop exists because ice is brighter than open water/land. Make that contrast
stronger or weaker (change $\alpha_i$, the frozen-planet albedo) and see what happens to
the width of the bistable range in $S$.

**Your task.**
1. For each $\alpha_i \in \{0.40, 0.45, 0.55\}$, sweep $S$ **up** from 1150 to 2000
   W m⁻² (cold branch) and **down** from 2000 to 1150 (warm branch), restarting from the
   previous equilibrium each step.
2. Use the given `threshold(T_series)` helper to find the $S$ at which each branch
   jumps; the bistable width is (deglaciation $S$) minus (re-freezing $S$).
3. Plot the three loops and print the three widths.

**Hints.**
* `S_grid = np.arange(1150.0, 2001.0, 10.0)`.
* `equilibrium_T(T, S=s, alphai=ai)` &mdash; note `co2` stays at its default 280.
* `threshold` returns `np.nan` if a branch never jumps within the grid &mdash; that is a
  real outcome for a strong enough feedback, not a bug.


In [ ]:
S_grid = np.arange(1150.0, 2001.0, 10.0)
alphai_list = [0.40, 0.45, 0.55]
widths = {}
loops = {}

def threshold(T_series, grid=S_grid, min_jump=15.0):
    """Grid value at the largest step in T_series, or nan if no step exceeds min_jump."""
    d = np.diff(np.asarray(T_series))
    return grid[np.argmax(d)] if d.size and d.max() > min_jump else np.nan

# ==== YOUR CODE ====
for ai in alphai_list:
    T_up, T = [], -60.0
    for s in S_grid:
        # T = equilibrium_T(T, S=s, alphai=ai); T_up.append(T)
        pass
    T_dn, T = [], 30.0
    for s in S_grid[::-1]:
        # T = equilibrium_T(T, S=s, alphai=ai); T_dn.append(T)
        pass
    T_up = np.array(T_up); T_dn = np.array(T_dn[::-1])
    loops[ai] = (T_up, T_dn)
    # widths[ai] = threshold(T_up) - threshold(T_dn)
    widths[ai] = np.nan
# ===================

for ai, w in widths.items():
    print(f"alpha_i = {ai}:  bistable width in S = {w} W/m2")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(8, 5))
for ai, colour in zip(alphai_list, ["tab:green", "tab:orange", "tab:purple"]):
    if ai in loops and len(loops[ai][0]):
        up, dn = loops[ai]
        ax.plot(S_grid, up, color=colour, lw=1)
        ax.plot(S_grid, dn, color=colour, lw=1, ls="--", label=f"$\\alpha_i$ = {ai}")
ax.axvline(1368, color="0.5", ls=":", label="present S")
ax.set_xlabel("solar constant S [W/m$^2$]"); ax.set_ylabel("equilibrium T [degC]")
ax.set_title("Hysteresis loop for three ice albedos (solid = up-sweep, dashed = down)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. Does a *stronger* ice&ndash;albedo contrast (larger $\alpha_i$) widen or narrow the
   bistable range? Explain in terms of how much the feedback amplifies a small cooling.
2. For a large enough $\alpha_i$, `threshold` returns `nan` for the up-sweep: the cold
   branch never deglaciates within the grid. What does that mean physically &mdash; can
   you always melt a Snowball by making the Sun brighter?
3. Clouds, dust on snow, and vegetation all change the *effective* $\alpha_i$. Which way
   would each push it?
```


## Exercise 3 &mdash; a seasonal snow&ndash;albedo feedback

The planetary feedback of this lecture has a small cousin that runs every spring in the
Himalaya: as snow melts, the ground darkens, absorbs more sun, and melts the rest
faster. Model one melt season.

**Background.** Over a melt season, drive a surface with a slowly rising energy input
$Q(t)$ (W m⁻²). The snow-covered fraction falls linearly with temperature,

$$f(T) = \operatorname{clip}\!\left(\frac{5 - T}{5}, 0, 1\right)
\quad(\text{1 at }0\,°\text{C},\ 0 \text{ at }5\,°\text{C}),$$

the surface albedo is $\alpha = f\,\alpha_\text{snow} + (1-f)\,\alpha_\text{ground}$ with
$\alpha_\text{snow} = 0.7$, $\alpha_\text{ground} = 0.2$, and the surface temperature
follows $c\,\dfrac{dT}{dt} = (1-\alpha)\,Q - \lambda\,T$ with $c = 8\times10^{6}$
J m⁻² K⁻¹ and $\lambda = 12$ W m⁻² K⁻¹.

**Your task.**
1. Complete `snow_fraction(T)` and the albedo line inside the loop.
2. Integrate for 150 days with `dt = 86400` s and $Q$ ramping from 120 to 240 W m⁻².
3. The plot shows $T(t)$, $f(t)$ and $\alpha(t)$. Identify when the melt "runs away".

**Hints.**
* `snow_fraction(T)` = `np.clip((5 - T) / 5, 0, 1)`.
* Inside the loop: `alpha = f * 0.7 + (1 - f) * 0.2`.
* `T = T + dt / c * ((1 - alpha) * Q[i] - lam * T)`.


In [ ]:
def snow_fraction(T):
    # ==== YOUR CODE ====
    return 1.0                      # <-- np.clip((5 - T) / 5, 0, 1)
    # ===================


days = 150
dt = 86400.0
Q = np.linspace(120.0, 240.0, days)      # rising energy input [W/m2]
c_surf, lam = 8e6, 12.0

T = -12.0
T_hist, f_hist, a_hist = [], [], []
for i in range(days):
    f = snow_fraction(T)
    # ==== YOUR CODE ====
    alpha = 0.5                     # <-- f * 0.7 + (1 - f) * 0.2
    # ===================
    T = T + dt / c_surf * ((1 - alpha) * Q[i] - lam * T)
    T_hist.append(T); f_hist.append(f); a_hist.append(alpha)

T_hist, f_hist, a_hist = map(np.array, (T_hist, f_hist, a_hist))
melt_start = np.argmax(f_hist < 0.99) if np.any(f_hist < 0.99) else -1
gone = np.argmax(f_hist <= 0.01) if np.any(f_hist <= 0.01) else -1
print(f"snow starts melting on day {melt_start}, fully gone by day {gone}")


In [ ]:
# --- plot (given) ---
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(T_hist, color="tab:red", label="surface T [degC]")
ax1.set_xlabel("day of melt season"); ax1.set_ylabel("T [degC]", color="tab:red")
ax1.axhline(0, color="0.7", lw=0.8)
ax2 = ax1.twinx()
ax2.plot(f_hist, color="tab:blue", label="snow fraction f")
ax2.plot(a_hist, color="tab:green", label="albedo")
ax2.set_ylabel("f  /  albedo"); ax2.set_ylim(0, 1)
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="center right")
ax1.set_title("One Himalayan melt season")
fig.tight_layout()


```{admonition} Answers
:class: note
1. Once melt starts, does the surface temperature rise *faster* than the forcing $Q$
   alone would explain? What is accelerating it?
2. Run it again from `T = -12.0` but with the snow albedo lowered from 0.7 to 0.55
   (dust/soot on snow). Does the snow disappear earlier? By how many days?
3. This is the same feedback as the planetary one in the lecture, but it does **not**
   produce two stable states here. Why not &mdash; what is different about a single
   seasonal cycle versus the equilibrium sweeps of Exercises 1&ndash;2?
```


## Where this goes next

* Multiple equilibria and hysteresis are the maths of **tipping points** (Lecture 2).
  The same picture describes the Atlantic overturning, ice sheets, and monsoon regimes.
* The seasonal snow&ndash;albedo feedback of Exercise 3 is why melting Himalayan
  snowpack is a concern for **Module 5** (impacts) and why the catchment snow module of
  the Lecture 7 exercise matters.

```{note} Sources
Combines the exercises of Lecture 5 with the *melting the Snowball Earth* problem from
the [*Climate of the Ocean*](https://github.com/florianboergel/climateoftheocean)
course (after Henri Drake, MIT 18.S191), for **CE524 Applied Hydroclimatology**.
Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
